# Tutorial: LM-01 Linear Regression

Audience:
- Learners studying the first linear-model lab in the curriculum.

Prerequisites:
- Matrix notation for linear regression, mean squared error, and train/test splits.
- Basic familiarity with polynomial features and regularization.

Learning goals:
- Fit linear regression models on synthetic data.
- Visualize underfitting, a good fit, and overfitting through polynomial regression.
- Compare ordinary least squares, ridge, and lasso on the same design matrix.
- Inspect how coefficient shrinkage changes as regularization strength increases.


## Outline

1. Generate a synthetic regression problem with a known target signal.
2. Fit low-, medium-, and high-degree polynomial regression models.
3. Compare train, test, and cross-validation error across degrees.
4. Compare ordinary least squares, ridge, and lasso on a rich polynomial basis.
5. Trace coefficient shrinkage as regularization increases.
6. Answer short follow-up questions.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEED = 11
rng = np.random.default_rng(SEED)
SEED


## Step 1 - Build a controlled regression problem

The target signal is known, so we can compare each fitted curve against the underlying truth instead of against noisy labels alone.


In [ ]:
def target_function(x: np.ndarray) -> np.ndarray:
    return np.sin(2.0 * np.pi * x) + 0.15 * np.cos(5.0 * np.pi * x)


def make_dataset(sample_size: int = 24, noise_std: float = 0.18, seed: int = SEED):
    local_rng = np.random.default_rng(seed)
    x_train = np.sort(local_rng.uniform(0.0, 1.0, size=sample_size)).reshape(-1, 1)
    y_train = target_function(x_train[:, 0]) + local_rng.normal(0.0, noise_std, size=sample_size)
    x_plot = np.linspace(0.0, 1.0, 400).reshape(-1, 1)
    y_true = target_function(x_plot[:, 0])
    return x_train, y_train, x_plot, y_true


x_train, y_train, x_plot, y_true = make_dataset()

fig, ax = plt.subplots(figsize=(6.8, 4.0))
ax.scatter(x_train[:, 0], y_train, color="black", s=24, label="training sample", zorder=3)
ax.plot(x_plot[:, 0], y_true, linestyle="--", color="tab:gray", label="true signal")
ax.set_title("Synthetic regression problem")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
fig.tight_layout()
plt.close(fig)

{"sample_size": len(x_train), "noise_std": 0.18}


## Step 2 - Fit polynomial regression at three complexity levels

A polynomial model is still linear in the coefficients. Increasing degree changes the feature map, not the fact that the predictor is linear in its parameters.


In [ ]:
def polynomial_pipeline(degree: int, model) -> Pipeline:
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("model", model),
        ]
    )


selected_degrees = {"underfit": 1, "good_fit": 4, "overfit": 12}
models = {
    label: polynomial_pipeline(degree, LinearRegression()).fit(x_train, y_train)
    for label, degree in selected_degrees.items()
}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.8), sharey=True)
for ax, (label, degree) in zip(axes, selected_degrees.items()):
    model = models[label]
    ax.scatter(x_train[:, 0], y_train, color="black", s=18, zorder=3)
    ax.plot(x_plot[:, 0], y_true, linestyle="--", color="tab:gray", label="true signal")
    ax.plot(x_plot[:, 0], model.predict(x_plot), color="tab:blue", label=f"degree {degree}")
    ax.set_title(label.replace("_", " ").title())
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
axes[-1].legend(loc="upper right")
fig.tight_layout()
plt.close(fig)

{
    label: {
        "degree": degree,
        "train_mse": float(mean_squared_error(y_train, model.predict(x_train))),
        "test_mse": float(mean_squared_error(y_true, model.predict(x_plot))),
    }
    for label, degree in selected_degrees.items()
    for model in [models[label]]
}


## Step 3 - Compare train, test, and cross-validation error across degree

Training error usually falls with model flexibility. Test and cross-validation error typically reach a minimum at an intermediate degree.


In [ ]:
degrees = list(range(1, 15))
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
sweep = []

for degree in degrees:
    model = polynomial_pipeline(degree, LinearRegression())
    model.fit(x_train, y_train)
    train_mse = mean_squared_error(y_train, model.predict(x_train))
    test_mse = mean_squared_error(y_true, model.predict(x_plot))
    cv_mse = -cross_val_score(
        polynomial_pipeline(degree, LinearRegression()),
        x_train,
        y_train,
        cv=cv,
        scoring="neg_mean_squared_error",
    ).mean()
    sweep.append({"degree": degree, "train_mse": train_mse, "test_mse": test_mse, "cv_mse": cv_mse})

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(degrees, [row["train_mse"] for row in sweep], marker="o", label="train MSE")
ax.plot(degrees, [row["test_mse"] for row in sweep], marker="o", label="test MSE")
ax.plot(degrees, [row["cv_mse"] for row in sweep], marker="o", label="5-fold CV MSE")
ax.set_title("Polynomial degree versus predictive error")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("Mean squared error")
ax.legend()
fig.tight_layout()
plt.close(fig)

sorted(sweep, key=lambda row: row["cv_mse"])[:5]


## Step 4 - Compare ordinary least squares, ridge, and lasso on a rich basis

Using a higher-degree basis makes regularization visible. The comparison is fair only if the polynomial features are standardized before the penalized models are fit.


In [ ]:
rich_degree = 12

ols_model = polynomial_pipeline(rich_degree, LinearRegression()).fit(x_train, y_train)
ridge_model = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=rich_degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=0.05)),
    ]
).fit(x_train, y_train)
lasso_model = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=rich_degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", Lasso(alpha=0.005, max_iter=50000)),
    ]
).fit(x_train, y_train)

comparison = {
    "OLS": ols_model,
    "Ridge": ridge_model,
    "Lasso": lasso_model,
}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.8), sharey=True)
for ax, (name, model) in zip(axes, comparison.items()):
    ax.scatter(x_train[:, 0], y_train, color="black", s=18, zorder=3)
    ax.plot(x_plot[:, 0], y_true, linestyle="--", color="tab:gray", label="true signal")
    ax.plot(x_plot[:, 0], model.predict(x_plot), color="tab:blue", label=name)
    ax.set_title(name)
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
axes[-1].legend(loc="upper right")
fig.tight_layout()
plt.close(fig)

{
    name: {
        "train_mse": float(mean_squared_error(y_train, model.predict(x_train))),
        "test_mse": float(mean_squared_error(y_true, model.predict(x_plot))),
    }
    for name, model in comparison.items()
}


## Step 5 - Trace coefficient shrinkage

Ridge usually shrinks coefficients smoothly toward zero. Lasso often drives some coefficients exactly to zero when the regularization is strong enough.


In [ ]:
poly = PolynomialFeatures(degree=rich_degree, include_bias=False)
X_poly = poly.fit_transform(x_train)
X_poly_plot = poly.transform(x_plot)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

alphas = np.geomspace(1e-4, 1.0, 12)
ridge_paths = []
lasso_paths = []

for alpha in alphas:
    ridge_fit = Ridge(alpha=alpha).fit(X_scaled, y_train)
    lasso_fit = Lasso(alpha=alpha, max_iter=50000).fit(X_scaled, y_train)
    ridge_paths.append(ridge_fit.coef_)
    lasso_paths.append(lasso_fit.coef_)

ridge_paths = np.vstack(ridge_paths)
lasso_paths = np.vstack(lasso_paths)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.0), sharey=True)
for idx in range(ridge_paths.shape[1]):
    axes[0].plot(alphas, ridge_paths[:, idx], linewidth=1.2)
    axes[1].plot(alphas, lasso_paths[:, idx], linewidth=1.2)

for ax, title in zip(axes, ["Ridge coefficient paths", "Lasso coefficient paths"]):
    ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel("alpha")

axes[0].set_ylabel("coefficient value")
fig.tight_layout()
plt.close(fig)

{
    "ridge_nonzero_last_alpha": int(np.count_nonzero(np.abs(ridge_paths[-1]) > 1e-8)),
    "lasso_nonzero_last_alpha": int(np.count_nonzero(np.abs(lasso_paths[-1]) > 1e-8)),
}


## Interpretation

Typical patterns in this notebook are:
- low polynomial degree underfits the oscillatory signal;
- an intermediate degree captures the trend without excessive variance;
- a high degree can drive training error down while worsening predictive error;
- ridge stabilizes the high-degree fit by shrinking all coefficients; and
- lasso tends to zero out some coefficients, giving a sparse approximation.

Try changing `sample_size`, `noise_std`, the selected degrees, or the regularization strengths to see how the bias-variance tradeoff changes.


## Exercises

- Increase the sample size from `24` to `60` and compare the overfit model before and after the change.
- Increase `noise_std` and record how the best cross-validation degree changes.
- Replace the sinusoidal target with a quadratic target and explain which polynomial degrees are now sufficient.
- Compare the number of nonzero lasso coefficients at the largest `alpha` with the ridge coefficients at the same `alpha`.


In [ ]:
# Exercise answer scaffold
def summarize_degree_search(results: list[dict[str, float]]) -> dict[str, float]:
    best_cv = min(results, key=lambda row: row["cv_mse"])
    return {
        "best_degree_by_cv": int(best_cv["degree"]),
        "best_cv_mse": float(best_cv["cv_mse"]),
        "lowest_training_mse_degree": int(min(results, key=lambda row: row["train_mse"])["degree"]),
    }


summarize_degree_search(sweep)
